In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls drive/MyDrive/lct

data_latest.csv  qwen3_0.6b-v1	synthetic_data.json


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# fourbit_models = [
#     "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
#     "unsloth/Qwen3-4B-unsloth-bnb-4bit",
#     "unsloth/Qwen3-8B-unsloth-bnb-4bit",
#     "unsloth/Qwen3-14B-unsloth-bnb-4bit",
#     "unsloth/Qwen3-32B-unsloth-bnb-4bit",

#     # 4bit dynamic quants for superior accuracy and low memory use
#     "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
#     "unsloth/Phi-4",
#     "unsloth/Llama-3.1-8B",
#     "unsloth/Llama-3.2-3B",
#     "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
# ] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B",
    max_seq_length = 4096,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.9: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2025.9.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


<a name="Data"></a>
### Data Prep
Qwen3 has both reasoning and a non reasoning mode. So, we should use 2 datasets:

1. We use the [Open Math Reasoning]() dataset which was used to win the [AIMO](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/leaderboard) (AI Mathematical Olympiad - Progress Prize 2) challenge! We sample 10% of verifiable reasoning traces that used DeepSeek R1, and whicht got > 95% accuracy.

2. We also leverage [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. But we need to convert it to HuggingFace's normal multiturn format as well.

In [ ]:
# from datasets import load_dataset
# reasoning_dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
# non_reasoning_dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

Let's see the structure of both datasets:

In [ ]:
# reasoning_dataset

In [ ]:
# non_reasoning_dataset

We now convert the reasoning dataset into conversational format:

In [ ]:
# def generate_conversation(examples):
#     problems  = examples["problem"]
#     solutions = examples["generated_solution"]
#     conversations = []
#     for problem, solution in zip(problems, solutions):
#         conversations.append([
#             {"role" : "user",      "content" : problem},
#             {"role" : "assistant", "content" : solution},
#         ])
#     return { "conversations": conversations, }

In [ ]:
# reasoning_conversations = tokenizer.apply_chat_template(
#     reasoning_dataset.map(generate_conversation, batched = True)["conversations"],
#     tokenize = False,
# )

Let's see the first transformed row:

In [ ]:
# reasoning_conversations[0]

In [ ]:
# @title
SYSTEM_PROMPT = """\
Твоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:
1.  Определить все продукты/услуги (темы), о которых упоминает клиент.
2.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.

**ВАЖНЫЕ ИНСТРУКЦИИ:**
-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.
-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).
-   **Строгость:** Не выдумывай темы. Если в отзыве нет явного упоминания продукта или услуги, не включай его.
-   **Символы `****`:** Это либо конфиденциальные данные (номера телефонов), либо ненормативная лексика. Учитывай общий контекст вокруг них для определения тональности.
-   **Если темы нет:** Если в отзыве невозможно определить ни одну тему, верни пустой массив `topic_sentiment_pairs`.

Вот список всех продуктов банка на сегодняшний день:
* **дебетовые карты:**
    * Умная дебетовая карта «Мир»
    * Премиальная карта Mir Supreme
    * Дебетовая карта с кэшбэком для самозанятых
    * Умная дебетовая карта «Мир»
    * Карта для автолюбителей «Газпромбанк—Газпромнефть»
    * Виртуальная дебетовая карта ГПБ&ФК «Зенит»
    * Дебетовая Пенсионная карта
* **Кредитные карты:**
    * Кредитная карта с льготным периодом до 120 дней
    * Простая кредитная карта
    * Кредитная карта 90 дней
    * Кредитная карта 180 дней Премиум
    * Кредитная карта для самозанятых
* **Накопительные счета:**
    * Накопительный счет
    * «Ежедневная выгода»
    * «Ежедневный процент»
    * «Премиум»
    * Социальный счет
* **Вклады:**
    * Вклад «Новые деньги»
    * Вклад «Ключевой момент»
    * Вклад «Копить»
    * Вклад «В Плюсе»
    * Вклад «Расширяй возможности»
    * Социальный вклад
* **Кредиты:**
    * Кредит наличными
    * Кредит наличными под залог недвижимости
    * Кредит на авто и другие цели
    * Рефинансирование потребительских кредитов
    * Дачный кредит
    * Кредит на образование
    * Кредит наличными для бюджетников
* **Другие услуги банка:**
    * Газпромбанк Мобайл
    * Газпромбанк Travel
    * Gazprom Pay
    * GorodPay
    * Газпром Бонус
    * Страховые и сервисные продукты
    * Депозитарные услуги


**Формат ответа — строго JSON:**
{
    "topic_sentiment_pairs": [
        {
            "topic": "Название темы 1",
            "sentiment": "positive/negative/neutral"
        },
        {
            "topic": "Название темы 2",
            "sentiment": "positive/negative/neutral"
        }
    ]
}


**Примеры:**
Отзыв 1: Мобильное приложение просто ужасное, постоянно вылетает. А вот вклад оформил недавно — условия хорошие.

Ответ:
{
    "topic_sentiment_pairs": [
        {
            "topic": "Мобильное приложение",
            "sentiment": "negative"
        },
        {
            "topic": "Вклады",
            "sentiment": "positive"
        }
    ]
}


Отзыв 2: Звонил узнать насчет ипотеки, но в поддержке ничем не помогли.

Ответ:
{
    "topic_sentiment_pairs": [
        {
            "topic": "Дистанционное обслуживание",
            "sentiment": "negative"
        },
        {
            "topic": "Ипотека",
            "sentiment": "neutral"
        }
    ]
},

**Вот отзывы клиентов, которые необходимо обработать:**
"""

Next we take the non reasoning dataset and convert it to conversational format as well.

We have to use Unsloth's `standardize_sharegpt` function to fix up the format of the dataset first.

In [ ]:
import json
import pandas as pd
import numpy as np

import glob
import os

In [ ]:
df = pd.read_csv("drive/MyDrive/lct/data_latest.csv")

df

,review_id,date,review_text,topic,subtopic,sentiment
0,1000087,2025-09-19,Вклад «Новые деньги» невозможно оформить без п...,Вклады,NaN,Negative
1,999494,2025-09-18,В июне 2025 года я порекомендовал премиальную ...,Дебетовые карты,NaN,Negative
2,999142,2025-09-17,Мошенниччиские аперации в интересах Ренессанс ...,Обслуживание,NaN,Negative
3,998360,2025-09-15,Купил услугу Газпром Бонус «Премиум» за 2 990 ...,Дебетовые карты,NaN,Negative
4,998516,2025-09-15,Производил оформление открытия срочного банков...,Вклады,«Накопительный»,Negative
...,...,...,...,...,...,...
4773,7470,2011-04-07,Ужастное обслуживание! Мало того потеряли доку...,Обслуживание,NaN,Negative
4774,7049,2011-03-28,Могут заблокировать рассчетную или кредитную к...,Кредитные карты,NaN,Negative
4775,5221,2011-01-25,"Мало того уже прошла неделя, а ПТС так и не ве...",Автокредиты,NaN,Negative
4776,5053,2011-01-16,Газпромбанк– отличный банк с отличными сотрудн...,Ипотека,NaN,Positive


In [ ]:
# Сдеалть разделение на train/test по дате
from datasets import Dataset

topics_sentiments_json = "drive/MyDrive/lct/synthetic_data.json"
original_reviews_csv = "drive/MyDrive/lct/data_latest.csv"

with open(topics_sentiments_json) as f:
    topics_sentiments_full = json.load(f)

topics_sentiments_pair = {
    int(topics_sentiments_full[i]["id"]) : topics_sentiments_full[i]["topic_sentiment_pairs"]
    for i in range(len(topics_sentiments_full))
}

original_reviews_df = pd.read_csv(original_reviews_csv)


user_prompts = []
assistant_answers = []

for i in range(original_reviews_df.shape[0]):
    original_review_series = original_reviews_df.iloc[i]
    review_id = original_review_series["review_id"]

    user_prompt = original_review_series["review_text"]

    assistant_answer = str(topics_sentiments_pair[review_id])

    user_prompts.append(user_prompt)
    assistant_answers.append(assistant_answer)


dataset_dict = {"user_prompt" : user_prompts, "assistant_answer" : assistant_answers}

dataset = Dataset.from_dict(dataset_dict)

In [ ]:
dataset = dataset.train_test_split(test_size=0.1)

dataset_train = dataset["train"]
dataset_test = dataset["test"]

In [ ]:
def convert_to_chatml(example):
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["user_prompt"]},
            {"role": "assistant", "content": example["assistant_answer"]}
        ]
    }

dataset_train = dataset_train.map(
    convert_to_chatml
)

dataset_test = dataset_test.map(
    convert_to_chatml
)

Map:   0%|          | 0/4300 [00:00<?, ? examples/s]

Map:   0%|          | 0/478 [00:00<?, ? examples/s]

In [ ]:
# from unsloth.chat_templates import standardize_sharegpt

# dataset_train = standardize_sharegpt(dataset_train)
# dataset_test = standardize_sharegpt(dataset_test)

# dataset_train = tokenizer.apply_chat_template(
#     dataset_train["conversations"],
#     tokenize = False,
# )

# dataset_test = tokenizer.apply_chat_template(
#     dataset_test["conversations"],
#     tokenize = False,
# )

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False) for convo in convos]
    return { "text" : texts, }

dataset_train = dataset_train.map(formatting_prompts_func, batched = True)

dataset_test = dataset_test.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/4300 [00:00<?, ? examples/s]

Map:   0%|          | 0/478 [00:00<?, ? examples/s]

In [ ]:
dataset_train["text"][10]

'<|im_start|>system\nТвоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:\n1.  Определить все продукты/услуги (темы), о которых упоминает клиент.\n2.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.\n\n**ВАЖНЫЕ ИНСТРУКЦИИ:**\n-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.\n-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).\n-   **Строгость:** Не выдумывай темы.

Let's see the first row

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = dataset_test, # Can set up evaluation!
    args = SFTConfig(
        do_eval=True,
        output_dir="drive/MyDrive/lct/qwen3_0.6b-v1",
        dataset_text_field = "text",
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 30,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        dataset_num_proc=4,
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/4300 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/478 [00:00<?, ? examples/s]

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
0.742 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,300 | Num Epochs = 1 | Total steps = 269
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 636,420,096 (6.34% trained)


Step,Training Loss
5,2.089600
10,1.865100
15,1.662500
20,1.519500
25,1.345100
30,1.236900
35,1.096500
40,0.907900
45,0.822300
50,0.750500


Unsloth: Will smartly offload gradients to save VRAM!


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

2476.6345 seconds used for training.
41.28 minutes used for training.
Peak reserved memory = 14.508 GB.
Peak reserved memory for training = 2.61 GB.
Peak reserved memory % of max memory = 98.419 %.
Peak reserved memory for training % of max memory = 17.706 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for reasoning inference are `temperature = 0.6, top_p = 0.95, top_k = 20`

For normal chat based inference, `temperature = 0.7, top_p = 0.8, top_k = 20`

In [ ]:
# messages = [
#     {"role" : "user", "content" : "Solve (x + 2)^2 = 0."}
# ]
# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize = False,
#     add_generation_prompt = True, # Must add for generation
#     enable_thinking = False, # Disable thinking
# )

# from transformers import TextStreamer
# _ = model.generate(
#     **tokenizer(text, return_tensors = "pt").to("cuda"),
#     max_new_tokens = 256, # Increase for longer outputs!
#     temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
#     streamer = TextStreamer(tokenizer, skip_prompt = True),
# )

In [ ]:
# messages = [
#     {"role" : "user", "content" : "Solve (x + 2)^2 = 0."}
# ]
# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize = False,
#     add_generation_prompt = True, # Must add for generation
#     enable_thinking = True, # Disable thinking
# )

# from transformers import TextStreamer
# _ = model.generate(
#     **tokenizer(text, return_tensors = "pt").to("cuda"),
#     max_new_tokens = 1024, # Increase for longer outputs!
#     temperature = 0.6, top_p = 0.95, top_k = 20, # For thinking
#     streamer = TextStreamer(tokenizer, skip_prompt = True),
# )

In [ ]:
from unsloth import FastLanguageModel

In [ ]:
model = FastLanguageModel.for_inference(model)

In [ ]:
from transformers import TextStreamer

test_idx = 27

print(dataset_test[test_idx]["user_prompt"])

messages = [
    [
        {'role': 'system','content' : SYSTEM_PROMPT},
        {"role" : 'user', 'content' : dataset_test[test_idx]["user_prompt"]}
    ]
]
text = tokenizer.apply_chat_template(
    messages[0],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

Если вы заказывали зарплатную карту в другом городе то перевыпустить её будет можно за деньги только и главное долго (по почте), до оператора практически не возможно дозвонится и минимальное ожидание более 30 минут, так же без ведома клиента подключают услугу овердрафт и если у вас есть долги у судебных приставов то вас загоняться в любой минус!


In [ ]:
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 125,
    temperature = 0.5, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

print(dataset_test[test_idx]["conversations"][-1]["content"])

<think>

</think>

[{'topic': 'Дебетовые карты', 'sentiment': 'negative'}, {'topic': 'Офисное обслуживание', 'sentiment': 'negative'}]<|im_end|>
[{'topic': 'Дебетовые карты', 'sentiment': 'negative'}, {'topic': 'Дистанционное обслуживание', 'sentiment': 'negative'}]


In [ ]:
model.save_pretrained("mount/models/gemma-3-270m-it-revews-fine-tune-v1")  # Local saving
tokenizer.save_pretrained("mount/models/gemma-3-270m-it-revews-fine-tune-v1")

('mount/models/gemma-3-270m-it-revews-fine-tune-v1/tokenizer_config.json',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/special_tokens_map.json',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/chat_template.jinja',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/vocab.json',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/merges.txt',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/added_tokens.json',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/tokenizer.json')

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/vocab.json',
 'lora_model/merges.txt',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False:
    model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False: # Pushing to HF Hub
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
